# Hospital Readmission Prediction — Initial Data Exploration

## Project Goal
Build a predictive model and decision-support dashboard that flags 30-day readmission risk before discharge, projecting $1.3M+ in annual savings for a mid-sized hospital admitting approximately 10,000 patients per year.

## Dataset
- **Source:** [UCI Machine Learning Repository — Diabetes 130-US Hospitals (1999–2008)](https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008)
- **Citation:** Strack, B., DeShazo, J. P., Gennings, C., Olmo, J. L., Ventura, S., Cios, K. J., & Clore, J. N. (2014). Impact of HbA1c measurement on hospital readmission rates: Analysis of 70,000 clinical database patient records. *BioMed Research International*, 2014.
- **Records:** 101,766 patient encounters
- **Features:** 50 columns spanning demographics, diagnoses, medications, and outcomes
- **Target:** 30-day readmission (binary classification)

## Notebook Structure
1. **Section 1:** Setup and data loading
2. **Section 2:** Initial data overview (shape, types, target distribution)
3. **Section 3:** Missing values analysis
4. **Section 4:** Feature engineering — binary target variable
5. **Section 5:** Save cleaned dataset

## Hypotheses (to test)
1. Older patients have higher readmission rates
2. Patients with longer length of stay are more likely to readmit
3. Patients with more prior emergency visits readmit more frequently
4. Discharge disposition strongly predicts readmission
5. Patients on insulin show different readmission patterns

---

**Author:** Satish Kumar Akrura
**Date:** May 13, 2026
**Status:** Day 4 of 6-week project

## Section 1: Setup and Data Loading

This section imports the required Python libraries and loads the diabetes hospital readmissions dataset for analysis.

In [6]:
# Import libraries for data analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ All libraries loaded successfully")
print(f"✓ pandas version: {pd.__version__}")
print(f"✓ numpy version: {np.__version__}")

✓ All libraries loaded successfully
✓ pandas version: 2.2.2
✓ numpy version: 1.26.4


In [8]:
# Load the dataset
df = pd.read_csv('../data/raw/diabetic_data.csv')

# Confirm it loaded
print(f"✓ Dataset loaded successfully")
print(f"  • Total rows: {df.shape[0]:,}")
print(f"  • Total columns: {df.shape[1]}")
print(f"  • Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

✓ Dataset loaded successfully
  • Total rows: 101,766
  • Total columns: 50
  • Memory usage: 192.87 MB


In [10]:
# Show the first 5 rows of the dataset
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


## Section 2: Initial Data Overview

Before cleaning missing values, let's understand:
- Data types of each column
- Distribution of the target variable (readmitted)
- Basic statistics

In [13]:
# Check data types and non-null counts for all columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

In [15]:
# Look at the target variable: readmitted
print("Readmission distribution (counts):")
print(df['readmitted'].value_counts())
print()
print("Readmission distribution (percentages):")
print(df['readmitted'].value_counts(normalize=True) * 100)

Readmission distribution (counts):
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Readmission distribution (percentages):
readmitted
NO     53.911916
>30    34.928169
<30    11.159916
Name: proportion, dtype: float64


## So only ~11% of patients readmit within 30 days.

## Section 3: Missing Values Analysis

This dataset stores missing values in two ways:
1. As the string `"?"` (for categorical columns)
2. As actual `NaN` (for some clinical measurements)

We need to convert `"?"` to proper `NaN` so pandas recognizes them correctly, then quantify missing data per column.

In [19]:
# Replace '?' with proper NaN so pandas recognizes them as missing
df = df.replace('?', np.nan)

# Confirm the replacement worked
print("✓ Replaced '?' with NaN")
print(f"  • Total missing values across all columns: {df.isnull().sum().sum():,}")

✓ Replaced '?' with NaN
  • Total missing values across all columns: 374,017


In [21]:
# Calculate missing values for each column
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

# Combine into a clean summary table
missing_summary = pd.DataFrame({
    'Missing Count': missing_count,
    'Missing %': missing_pct.round(2)
})

# Show only columns with missing values, sorted by % missing
missing_summary = missing_summary[missing_summary['Missing Count'] > 0]
missing_summary = missing_summary.sort_values('Missing %', ascending=False)

print(f"Columns with missing values: {len(missing_summary)}\n")
print(missing_summary)

Columns with missing values: 9

                   Missing Count  Missing %
weight                     98569      96.86
max_glu_serum              96420      94.75
A1Cresult                  84748      83.28
medical_specialty          49949      49.08
payer_code                 40256      39.56
race                        2273       2.23
diag_3                      1423       1.40
diag_2                       358       0.35
diag_1                        21       0.02


## Section 3.1: Missing Values Decisions

Based on the missing values audit above, here are my documented decisions for each column:

### Drop entirely (too much missing to be useful)
| Column | % Missing | Rationale |
|--------|-----------|-----------|
| `weight` | 96.86% | Imputing 99K values would introduce massive noise |
| `max_glu_serum` | 94.75% | Lab test rarely ordered; not enough signal to retain |

### Keep with "Unknown" imputation (missingness itself is informative)
| Column | % Missing | Rationale |
|--------|-----------|-----------|
| `A1Cresult` | 83.28% | Missing = "Test not ordered" — clinically meaningful absence |
| `medical_specialty` | 49.08% | Specialty type matters; missing may indicate ER/general intake |
| `payer_code` | 39.56% | Payer type predicts care patterns; missing is its own category |

### Drop rows (very small % missing)
| Column | % Missing | Rationale |
|--------|-----------|-----------|
| `race` | 2.23% | Acceptable row loss |
| `diag_3`, `diag_2`, `diag_1` | < 2% combined | Diagnosis codes are critical — drop incomplete rows |

**Estimated final dataset:** ~97,500 rows × 48 columns

In [24]:
# Drop columns with too much missing data
columns_to_drop = ['weight', 'max_glu_serum']
df = df.drop(columns=columns_to_drop)

print(f"✓ Dropped columns: {columns_to_drop}")
print(f"  • New shape: {df.shape}")

✓ Dropped columns: ['weight', 'max_glu_serum']
  • New shape: (101766, 48)


In [26]:
# Impute moderately-missing columns with "Unknown"
columns_to_impute = ['A1Cresult', 'medical_specialty', 'payer_code']

for col in columns_to_impute:
    df[col] = df[col].fillna('Unknown')
    print(f"✓ Imputed '{col}' missing values as 'Unknown'")

print(f"\n  • Missing values remaining: {df[columns_to_impute].isnull().sum().sum()}")

✓ Imputed 'A1Cresult' missing values as 'Unknown'
✓ Imputed 'medical_specialty' missing values as 'Unknown'
✓ Imputed 'payer_code' missing values as 'Unknown'

  • Missing values remaining: 0


In [28]:
# Drop rows missing race or diagnosis codes
rows_before = len(df)
df = df.dropna(subset=['race', 'diag_1', 'diag_2', 'diag_3'])
rows_after = len(df)
rows_dropped = rows_before - rows_after

print(f"✓ Dropped {rows_dropped:,} rows with missing race or diagnosis codes")
print(f"  • Rows before: {rows_before:,}")
print(f"  • Rows after: {rows_after:,}")
print(f"  • Percentage retained: {(rows_after/rows_before)*100:.2f}%")

✓ Dropped 3,713 rows with missing race or diagnosis codes
  • Rows before: 101,766
  • Rows after: 98,053
  • Percentage retained: 96.35%


In [30]:
# Final missing values check
remaining_missing = df.isnull().sum().sum()

print(f"Final missing values check:")
print(f"  • Total missing values remaining: {remaining_missing}")
print(f"  • Final dataset shape: {df.shape}")
print(f"  • Total rows × columns: {df.shape[0]:,} × {df.shape[1]}")

if remaining_missing == 0:
    print("\n✅ All missing values resolved!")
else:
    print(f"\n⚠️ Still {remaining_missing} missing values to handle.")

Final missing values check:
  • Total missing values remaining: 0
  • Final dataset shape: (98053, 48)
  • Total rows × columns: 98,053 × 48

✅ All missing values resolved!


## Section 4: Feature Engineering — Binary Target Variable

The original `readmitted` column has three values: `NO`, `>30`, `<30`.

For HRRP-focused prediction, we need a binary target:
- **`readmitted_30d` = 1** → Patient WAS readmitted within 30 days (the `<30` group)
- **`readmitted_30d` = 0** → Patient was NOT readmitted within 30 days (everything else)

This aligns with how CMS measures readmissions for the Hospital Readmissions Reduction Program.

In [33]:
# Create binary target: 1 if readmitted within 30 days, 0 otherwise
df['readmitted_30d'] = (df['readmitted'] == '<30').astype(int)

# Verify the new column
print("✓ Created binary target column: readmitted_30d\n")

print("Distribution of new binary target:")
print(df['readmitted_30d'].value_counts())
print()

print("Percentage distribution:")
print((df['readmitted_30d'].value_counts(normalize=True) * 100).round(2))

✓ Created binary target column: readmitted_30d

Distribution of new binary target:
readmitted_30d
0    86987
1    11066
Name: count, dtype: int64

Percentage distribution:
readmitted_30d
0    88.71
1    11.29
Name: proportion, dtype: float64


In [35]:
# Cross-check: show original readmitted column vs new binary target
print("Cross-tabulation of original vs binary target:")
print(pd.crosstab(df['readmitted'], df['readmitted_30d'], margins=True))

Cross-tabulation of original vs binary target:
readmitted_30d      0      1    All
readmitted                         
<30                 0  11066  11066
>30             34649      0  34649
NO              52338      0  52338
All             86987  11066  98053


### Critical Insight: Class Imbalance

The target variable is heavily imbalanced:
- **88.71% of patients (86,987)** are NOT readmitted within 30 days
- **11.29% of patients (11,066)** ARE readmitted within 30 days

This 8:1 imbalance is realistic and matches healthcare industry benchmarks. National HRRP-tracked condition readmission rates typically range from 10-15%.

**Implications for Modeling (Week 3):**

| Concern | Strategy |
|---------|----------|
| Accuracy will be misleading | A "predict 0 for everyone" model would score 88.71% — useless for our goal |
| Need to catch readmissions, not over-predict | Prioritize **Recall** over Precision |
| Class imbalance distorts training | Use **SMOTE** or **class_weight='balanced'** in scikit-learn |
| Multiple metrics needed | Track **ROC-AUC**, **Precision-Recall curve**, and **F1-score** |

**Why this matters for the business case:**

The hospital cost of missing a readmission (false negative) is approximately **15,200 USD per case**. The cost of an unneeded intervention (false positive) is approximately **500 USD**. This 30:1 cost ratio further reinforces optimizing for recall over precision.

In [43]:
# Save the cleaned dataset for use in future notebooks
output_path = '../data/processed/cleaned_data.csv'
df.to_csv(output_path, index=False)

print(f"✓ Cleaned dataset saved to: {output_path}")
print(f"  • Final shape: {df.shape}")
print(f"  • File ready for Week 2 EDA and beyond")

✓ Cleaned dataset saved to: ../data/processed/cleaned_data.csv
  • Final shape: (98053, 49)
  • File ready for Week 2 EDA and beyond


In [45]:
# Day 4 final summary
print("=" * 60)
print("DAY 4 COMPLETE — DATA CLEANING SUMMARY")
print("=" * 60)
print()
print(f"Original dataset:    101,766 rows × 50 columns")
print(f"Cleaned dataset:     {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Rows retained:       {(df.shape[0]/101766)*100:.2f}%")
print()
print("Cleaning actions taken:")
print("  • Dropped 2 columns (weight, max_glu_serum) — too much missing data")
print("  • Imputed 'Unknown' for 3 columns (A1Cresult, medical_specialty, payer_code)")
print("  • Dropped 3,713 rows with missing race or diagnosis codes")
print("  • Created binary target: readmitted_30d")
print()
print("Target distribution:")
print(f"  • Class 0 (NOT readmitted in 30d): {(df['readmitted_30d']==0).sum():,} ({(df['readmitted_30d']==0).mean()*100:.2f}%)")
print(f"  • Class 1 (Readmitted within 30d): {(df['readmitted_30d']==1).sum():,} ({(df['readmitted_30d']==1).mean()*100:.2f}%)")
print()
print("Ready for Week 2: Exploratory Data Analysis (EDA)")
print("=" * 60)

DAY 4 COMPLETE — DATA CLEANING SUMMARY

Original dataset:    101,766 rows × 50 columns
Cleaned dataset:     98,053 rows × 49 columns
Rows retained:       96.35%

Cleaning actions taken:
  • Dropped 2 columns (weight, max_glu_serum) — too much missing data
  • Imputed 'Unknown' for 3 columns (A1Cresult, medical_specialty, payer_code)
  • Dropped 3,713 rows with missing race or diagnosis codes
  • Created binary target: readmitted_30d

Target distribution:
  • Class 0 (NOT readmitted in 30d): 86,987 (88.71%)
  • Class 1 (Readmitted within 30d): 11,066 (11.29%)

Ready for Week 2: Exploratory Data Analysis (EDA)
